In [20]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import make_pipeline
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report, confusion_matrix

df = pd.read_csv('./data/spam.csv', encoding='latin-1')[['v1','v2']]
df.columns = ['label', 'text']

print(df.shape)
print(df['label'].value_counts())
df.head()

(5572, 2)
label
ham     4825
spam     747
Name: count, dtype: int64


,label,text
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


In [21]:
X_train, X_test, y_train, y_test = train_test_split(df['text'], df['label'], test_size=0.2, random_state=42, stratify=df['label'])

In [22]:
model_cv = make_pipeline(CountVectorizer(stop_words='english'), MultinomialNB())
model_cv.fit(X_train, y_train)
y_pred_cv = model_cv.predict(X_test)

print(classification_report(y_test, y_pred_cv))
print(confusion_matrix(y_test, y_pred_cv))

              precision    recall  f1-score   support

         ham       0.99      0.99      0.99       966
        spam       0.96      0.92      0.94       149

    accuracy                           0.98      1115
   macro avg       0.97      0.96      0.96      1115
weighted avg       0.98      0.98      0.98      1115

[[960   6]
 [ 12 137]]


In [23]:
model_tfidf = make_pipeline(TfidfVectorizer(stop_words='english'),MultinomialNB())
model_tfidf.fit(X_train, y_train)
y_pred_tfidf = model_tfidf.predict(X_test)

print(classification_report(y_test, y_pred_tfidf))
print(confusion_matrix(y_test, y_pred_tfidf))

              precision    recall  f1-score   support

         ham       0.97      1.00      0.98       966
        spam       1.00      0.77      0.87       149

    accuracy                           0.97      1115
   macro avg       0.98      0.88      0.92      1115
weighted avg       0.97      0.97      0.97      1115

[[966   0]
 [ 35 114]]


In [24]:
def ringkas(nama, pred):
    return {
        'fitur': nama,
        'accuracy': accuracy_score(y_test, pred),
        'precision': precision_score(y_test, pred, pos_label='spam'),
        'recall': recall_score(y_test, pred, pos_label='spam'),
        'f1': f1_score(y_test, pred, pos_label='spam')
    }

hasil = pd.DataFrame([ringkas('CV', y_pred_cv), ringkas('TF-IDF', y_pred_tfidf)])
print(hasil.head())

    fitur  accuracy  precision    recall        f1
0      CV  0.983857   0.958042  0.919463  0.938356
1  TF-IDF  0.968610   1.000000  0.765101  0.866920


In [25]:
y_bin = (df['label'] == 'spam').astype(int)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

for nama, vec in [('CV', CountVectorizer), ('TF-IDF', TfidfVectorizer)]:
    pipe = make_pipeline(vec(stop_words='english'), MultinomialNB())
    acc = cross_val_score(pipe, df['text'], y_bin, cv=cv, scoring='accuracy')
    f1 = cross_val_score(pipe, df['text'], y_bin, cv=cv, scoring='f1')
    print(f"{nama:16s} Accuracy = {acc.mean():.4f} | F1 spam = {f1.mean():.4f}")

CV               Accuracy = 0.9856 | F1 spam = 0.9457
TF-IDF           Accuracy = 0.9709 | F1 spam = 0.8784


Kesimpulan: dari kedua fitur tersebut, fitur yang terbaik adalah fitur CV karena keseimbangan nilai antara precision recall & f1. TF-IDF memiliki precision yang sempurna, tetapi recall spam lebih rendah.